# 089 — GAN y entrenamiento adversarial

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Pérdidas del juego

L_D = −log 0.8 − log 0.7 = 0.223 + 0.357 = **0.580**.
L_G = −log 0.3 = **1.204**. D está relativamente cómodo; G aún recibe señal
fuerte porque sus muestras se detectan con facilidad.


In [ ]:
import math

d_real, d_fake = 0.8, 0.3
l_d = -math.log(d_real) - math.log(1 - d_fake)
l_g = -math.log(d_fake)
print(f"L_D = {l_d:.3f}, L_G = {l_g:.3f}")


## Solución 2 — Discriminador óptimo

D*(x) = 0.6 / 0.8 = **0.75**. Cuando p_g = p_data, D*(x) = ½ en todo punto: el
discriminador óptimo ya no distingue mejor que el azar, y el valor del juego alcanza
−log 4, que es el mínimo de la divergencia Jensen-Shannon (JS = 0). Ese es el
equilibrio teórico de Goodfellow et al. (2014).


In [ ]:
p_data, p_g = 0.6, 0.2
d_star = p_data / (p_data + p_g)
print(f"D* = {d_star}")  # 0.75; en equilibrio p_g=p_data -> D*=0.5


## Solución 3 — Detectar colapso de modos

Sí: el generador cubre esencialmente 2 de 10 modos. La entropía observada
H ≈ 0.30 bits (tratando "resto" como una clase de p = 0.01) frente al máximo
log₂10 ≈ 3.32 bits para cobertura uniforme. La nitidez y la pérdida baja no miden
**cobertura**: por eso se usan métricas de diversidad (entropía de clases, recall
en precision/recall generativo, FID sobre muestras grandes).


In [ ]:
import math

probs = [480/1000, 510/1000, 10/1000]
h = -sum(p * math.log2(p) for p in probs)
print(f"H observada ≈ {h:.2f} bits vs máximo {math.log2(10):.2f} bits")


## Solución 4 — Contrato del laboratorio

La semilla del laboratorio fija toda la trayectoria pseudoaleatoria y hace el
resultado reproducible; el z de una GAN también nace de un generador pseudoaleatorio,
pero su papel es ser la **fuente de diversidad** del modelo, no un mecanismo de
auditoría. Se parecen en el mecanismo (PRNG), difieren en el propósito
(reproducibilidad del experimento vs variedad de las muestras).


In [ ]:
result = run_lab("generation", seed=89)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)
